In [5]:
import csv
import numpy as np


## 1- Read CSV File

In [ ]:
articles = []
with open('articles.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        articles.append(row)

## 2- Clean Articles Content

In [25]:
def clean_text(content):
    cleaned_content = ""
    content = content.lower()

    for char in content:
        if char.isalpha() or char == " ":
            cleaned_content += char

    words = cleaned_content.split()

    return words


for article in articles:
    article["tokens"] = clean_text(article["content"])


## 3- Building Bag of Words 


In [29]:
vocabulary = []

for article in articles:
    for word in article["tokens"]:
        if word not in vocabulary:
            vocabulary.append(word)
        
print(vocabulary)

['artificial', 'intelligence', 'is', 'a', 'field', 'of', 'computer', 'science', 'that', 'focuses', 'on', 'creating', 'systems', 'capable', 'performing', 'tasks', 'normally', 'require', 'human', 'such', 'as', 'learning', 'reasoning', 'and', 'problem', 'solving', 'machine', 'subset', 'enables', 'to', 'learn', 'from', 'data', 'improve', 'their', 'performance', 'without', 'being', 'explicitly', 'programmed', 'deep', 'specialized', 'branch', 'uses', 'neural', 'networks', 'with', 'many', 'layers', 'analyze', 'complex', 'patterns', 'in', 'large', 'datasets', 'robotics', 'advancing', 'rapidly', 'by', 'integrating', 'create', 'autonomous', 'machines', 'interacting', 'humans', 'combines', 'statistics', 'programming', 'domain', 'knowledge', 'extract', 'useful', 'insights', 'structured', 'unstructured', 'big', 'technologies', 'are', 'designed', 'process', 'massive', 'volumes', 'efficiently', 'using', 'distributed', 'computing', 'cybersecurity', 'protecting', 'cyber', 'attacks', 'unauthorized', 'ac

## 4- Building Vector Representation

In [30]:
for article in articles:
    vector = []
    
    for word in vocabulary:
        if word in article["tokens"]:
            vector.append(1)
        else:
            vector.append(0)
            
    article["vector"] = vector

## 5- Calculation of Cosine Similarities

In [31]:
vectors = []

for article in articles:
    vectors.append(np.array(article["vector"]))

In [32]:
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    # to avoid division by zero
    if norm1 == 0 or norm2 == 0:
        return 0
    
    return dot_product / (norm1 * norm2)

## 6- Similarity Matrix

In [33]:
n = len(vectors)

similarity_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        similarity_matrix[i][j] = cosine_similarity(vectors[i], vectors[j])

In [36]:
similarity_matrix

array([[1.        , 0.36803737, 0.2090605 , 0.34236839, 0.10127394,
        0.14269545, 0.24514517, 0.1902606 , 0.09805807, 0.15191091],
       [0.36803737, 1.        , 0.31118796, 0.27300945, 0.21535276,
        0.25286087, 0.20851441, 0.10114435, 0.20851441, 0.21535276],
       [0.2090605 , 0.31118796, 1.        , 0.23262105, 0.05504819,
        0.15512631, 0.05330018, 0.05170877, 0.15990054, 0.11009638],
       [0.34236839, 0.27300945, 0.23262105, 1.        , 0.11268723,
        0.15877684, 0.05455447, 0.10585122, 0.16366342, 0.16903085],
       [0.10127394, 0.21535276, 0.05504819, 0.11268723, 1.        ,
        0.18786729, 0.19364917, 0.12524486, 0.19364917, 0.13333333],
       [0.14269545, 0.25286087, 0.15512631, 0.15877684, 0.18786729,
        1.        , 0.18190172, 0.17647059, 0.30316953, 0.25048972],
       [0.24514517, 0.20851441, 0.05330018, 0.05455447, 0.19364917,
        0.18190172, 1.        , 0.18190172, 0.125     , 0.12909944],
       [0.1902606 , 0.10114435, 0.0517087

### saving the matrix to pkl file

In [37]:
import pickle

with open("similarities.pkl", "wb") as f:
    pickle.dump(similarity_matrix, f)

## 8- Finding Most Similar Articles

In [43]:
def get_top_3_similar(article_id):
    
    index = None
    for i, article in enumerate(articles):
        if article["id"] == str(article_id):
            index = i
            break
    
    if index is None:
        return "Article not found"
    
    similarities = similarity_matrix[index]
    
    indexed_similarities = list(enumerate(similarities))
    
    indexed_similarities = [x for x in indexed_similarities if x[0] != index]
    
    indexed_similarities.sort(key=lambda x: x[1], reverse=True)
    
    top_3 = indexed_similarities[:3]
    
    result = []
    for idx, score in top_3:
        result.append(articles[idx]["title"])
        
    return result

In [45]:
get_top_3_similar(5)

['Machine Learning Basics',
 'Cybersecurity Fundamentals',
 'Internet of Things Overview']